# DetectIA — Fase 1: Entrenamiento Rápido SVM (Normalizado en Google Colab)

> **Objetivo:** Entrenar el clasificador de vectores de soporte (**SVM Lineal**) con el dataset MNIST (`digitos.csv`) usando `SVC(kernel='linear')`, con **normalización de píxeles (`/ 255.0`)**, diagnóstico de **aprendizaje vs. memorización (Train/Test)**, **curva de aprendizaje** y análisis **por épocas**.

---
### Pasos de ejecución:
1. Ve a **Entorno de ejecución** > **Ejecutar todas** (o presiona `Ctrl + F9`).
2. En la **Celda 2**, sube el archivo `digitos.csv` cuando te lo solicite (si no lo arrastraste antes a la barra izquierda).
3. Al finalizar, se descargará automáticamente `artefactos_detectia.zip` con el modelo, métricas y todos los gráficos (incluidas las curvas de aprendizaje y épocas).

### Celda 1: Preparación del Entorno y Versiones
Fijamos `scikit-learn 1.9.1` para mantener compatibilidad del archivo `.joblib` entre Google Colab y el entorno local.

In [ ]:
!pip install -q --upgrade scikit-learn==1.9.1 joblib matplotlib pandas numpy

import sys
import os
import json
import time
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib

from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, hinge_loss

print("[OK] Entorno listo en Google Colab:")
print(f" - Python:       {sys.version.split()[0]}")
print(f" - scikit-learn: {sklearn.__version__}")
print(f" - joblib:       {joblib.__version__}")
print(f" - numpy:        {np.__version__}")

### Celda 2: Carga del Dataset `digitos.csv`

In [ ]:
from google.colab import files

nombre_archivo = 'digitos.csv'

if not os.path.exists(nombre_archivo):
    print("[AVISO] 'digitos.csv' no encontrado en Colab.")
    print("Por favor, selecciona 'digitos.csv' desde tu computadora:")
    subidos = files.upload()

print("\nLeyendo dataset 'digitos.csv' (42.000 filas x 785 columnas)...")
df = pd.read_csv(nombre_archivo)
print(f"[OK] Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas.")

conteo = df['label'].value_counts().sort_index()
print("\nDistribución por dígito:")
print(conteo)

### Celda 3: Visualización de Muestras e Histograma

In [ ]:
os.makedirs('img', exist_ok=True)

# 1. Histograma de distribución
plt.figure(figsize=(9, 4))
conteo.plot(kind='bar', color='#2563eb', edgecolor='black')
plt.title('Distribución de Clases en MNIST (Dígitos 0 al 9)', fontsize=13, fontweight='bold')
plt.xlabel('Dígito')
plt.ylabel('Cantidad de Muestras')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('img/histograma_distribucion.png', dpi=150)
plt.show()

# 2. Rejilla con 16 dígitos de muestra
plt.figure(figsize=(7, 7))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    matriz = df.iloc[i, 1:785].values.reshape(28, 28)
    plt.imshow(matriz, cmap='gray')
    plt.title(f"Clase: {df.iloc[i, 0]}", fontsize=10)
    plt.axis('off')
plt.suptitle('Muestras del dataset MNIST (28x28)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('img/muestra_digitos.png', dpi=150)
plt.show()

print("[OK] Gráficos iniciales guardados en 'img/'.")

### Celda 4: Partición de Datos y Normalización (`/ 255.0`)
Dividimos en **80% entrenamiento** y **20% prueba** con semilla `42` (estratificado para mantener el balance entre dígitos) y normalizamos los píxeles a escala `[0.0, 1.0]`.

In [ ]:
SEMILLA = 42

pixeles = df.iloc[:, 1:785]
digitos = df.iloc[:, 0:1]

X_train, X_test, y_train, y_test = train_test_split(
    pixeles,
    digitos,
    test_size=0.20,
    random_state=SEMILLA,
    stratify=digitos
)

# Normalizamos dividiendo entre 255.0 para acelerar la convergencia matemática del SVM
X_train_norm = X_train.values.astype(np.float32) / 255.0
X_test_norm = X_test.values.astype(np.float32) / 255.0

y_train_arr = y_train.values.ravel()
y_test_arr = y_test.values.ravel()

print(f"[OK] Partición completada (random_state={SEMILLA}):")
print(f" - Entrenamiento : {X_train_norm.shape[0]} muestras x {X_train_norm.shape[1]} atributos (Rango: [0.0, 1.0])")
print(f" - Evaluación    : {X_test_norm.shape[0]} muestras x {X_test_norm.shape[1]} atributos (Rango: [0.0, 1.0])")

### Celda 5: Entrenamiento del Modelo SVM Lineal (`SVC`)
`SVC(kernel='linear')` resuelve una optimización convexa exacta (**no usa épocas**, converge al óptimo matemático global).

In [ ]:
modelo = SVC(kernel='linear', C=0.05, cache_size=2000, random_state=SEMILLA)

print("Iniciando entrenamiento del modelo SVM Lineal (Normalizado / 255.0)...")

inicio_train = time.time()
modelo.fit(X_train_norm, y_train_arr)
tiempo_train = time.time() - inicio_train

print(f"\n[OK] Entrenamiento completado en {tiempo_train:.2f} segundos ({tiempo_train/60:.2f} minutos).")

### Celda 6: Diagnóstico Train vs. Test, Precisión y Matriz de Confusión
Comparamos la precisión en **Entrenamiento (Train)** frente a **Prueba (Test)** para comprobar si el modelo generaliza bien o está memorizando (*overfitting*).

In [ ]:
print(f"Evaluando sobre Train ({len(X_train_norm)} muestras) y Test ({len(X_test_norm)} muestras)...")
y_pred_train = modelo.predict(X_train_norm)
precision_train = accuracy_score(y_train_arr, y_pred_train)

inicio_pred = time.time()
y_pred = modelo.predict(X_test_norm)
tiempo_pred = time.time() - inicio_pred

precision = accuracy_score(y_test_arr, y_pred)
brecha = (precision_train - precision) * 100.0

print(f"Predicción en Test finalizada en {tiempo_pred:.2f} segundos.")
print("=" * 55)
print(f"Accuracy en TRAIN : {precision_train * 100:.2f}%")
print(f"Accuracy en TEST  : {precision * 100:.2f}%")
print(f"Brecha (Gap)      : {brecha:.2f} puntos porcentuales")
if brecha > 5.0:
    print("[ADVERTENCIA] Posible sobreajuste (memorización): Train está muy por encima de Test.")
else:
    print("[OK] Generalización saludable: Train y Test avanzan de la mano sin memorizar en exceso.")
print("=" * 55)

print("\n--- Reporte de Clasificación por Dígito (Test) ---")
print(classification_report(y_test_arr, y_pred, digits=4))

# Matriz de Confusión
conf = confusion_matrix(y_test_arr, y_pred)
plt.figure(figsize=(9, 7))
plt.imshow(conf, cmap=plt.cm.Blues)
plt.title(f"Matriz de Confusión — SVM Lineal (Test Accuracy: {precision*100:.2f}%)", fontsize=12, fontweight='bold')
plt.colorbar()
classes = list(range(10))
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)
thresh = conf.max() / 2.0
for fila in range(conf.shape[0]):
    for col in range(conf.shape[1]):
        val = conf[fila, col]
        color = 'white' if val > thresh else 'black'
        plt.text(col, fila, f"{val}", ha='center', va='center', color=color, fontsize=9)
plt.xlabel('Valores Predichos', fontweight='bold')
plt.ylabel('Valores Reales', fontweight='bold')
plt.tight_layout()
plt.savefig('img/matriz_confusion.png', dpi=150)
plt.show()

### Celda 7: Curva de Aprendizaje de `SVC` (Train vs. Validación por Tamaño de Datos)
Muestra cómo evoluciona la exactitud en **Train** y en **Validación** conforme el modelo recibe más ejemplos (10%, 25%, 50%, 75% y 100% del set de entrenamiento). Si ambas curvas se acercan entre sí, el modelo aprende patrones reales; si Train queda en ~100% y Validación mucho más abajo, hay memorización.

In [ ]:
print("Calculando curva de aprendizaje de SVC (4 tamaños x 2 folds, con progreso)...")

tamanos, scores_train, scores_val = learning_curve(
    SVC(kernel='linear', C=0.05, cache_size=2000, random_state=SEMILLA),
    X_train_norm,
    y_train_arr,
    train_sizes=np.array([0.15, 0.40, 0.70, 1.0]),
    cv=2,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

train_mean = scores_train.mean(axis=1) * 100
train_std = scores_train.std(axis=1) * 100
val_mean = scores_val.mean(axis=1) * 100
val_std = scores_val.std(axis=1) * 100

plt.figure(figsize=(9, 5))
plt.plot(tamanos, train_mean, 'o-', color='#1a73e8', linewidth=2.2, label='Entrenamiento (Train)')
plt.fill_between(tamanos, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#1a73e8')

plt.plot(tamanos, val_mean, 's-', color='#1e8e3e', linewidth=2.2, label='Validación (Test / CV)')
plt.fill_between(tamanos, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#1e8e3e')

for x_pt, tr_pt, va_pt in zip(tamanos, train_mean, val_mean):
    plt.annotate(f"{tr_pt:.1f}%", (x_pt, tr_pt), textcoords="offset points", xytext=(0, 8), ha='center', fontsize=8.5, color='#1a73e8')
    plt.annotate(f"{va_pt:.1f}%", (x_pt, va_pt), textcoords="offset points", xytext=(0, -14), ha='center', fontsize=8.5, color='#1e8e3e')

plt.title('Curva de Aprendizaje — SVC Lineal (Train vs. Validación)', fontsize=13, fontweight='bold')
plt.xlabel('Cantidad de muestras de entrenamiento')
plt.ylabel('Exactitud / Accuracy (%)')
plt.ylim(80, 101)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('img/curva_aprendizaje_svc.png', dpi=150)
plt.show()
print("[OK] Guardada en 'img/curva_aprendizaje_svc.png'.")

### Celda 8: Entrenamiento por Épocas (Detector de Tope de Aprendizaje y Overfitting)
Como `SVC` no entrena por épocas, aquí entrenamos el **mismo modelo SVM Lineal mediante Descenso de Gradiente Estocástico** (`SGDClassifier(loss='hinge')`) usando `partial_fit()` **época por época**.

Puedes ajustar `MAX_EPOCAS` (ej. `50`, `80`, `100`) y `PACIENCIA` para ver exactamente:
- En qué época **alcanza su máximo en Test**.
- A partir de qué época **deja de aprender** (se estanca o la curva de Train sigue subiendo mientras Test baja/se aplana).

In [ ]:
# Parámetros configurables
MAX_EPOCAS = 50       # Aumentar a 80 o 100 para observar el estancamiento
PACIENCIA = 8         # Épocas sin mejora en Test antes de marcar 'tope alcanzado'
TOL_MEJORA = 0.0005   # Mejora mínima (0.05%) para considerarse avance real

svm_epocas = SGDClassifier(
    loss='hinge',             # Función de pérdida de SVM Lineal
    alpha=0.0002,             # Regularización L2 suave para evitar sobreajuste
    learning_rate='invscaling',
    eta0=0.05,
    power_t=0.25,
    random_state=SEMILLA
)

clases = np.unique(y_train_arr)
rng = np.random.default_rng(SEMILLA)

hist_train_acc, hist_test_acc = [], []
hist_train_loss, hist_test_loss = [], []

mejor_test_acc = -1.0
mejor_epoca = 1
sin_mejora = 0
epoca_tope = None

print(f"Entrenando SVM Lineal por épocas (máximo {MAX_EPOCAS} épocas)...")
for epoca in range(1, MAX_EPOCAS + 1):
    # Barajar los datos en cada época
    perm = rng.permutation(len(X_train_norm))
    svm_epocas.partial_fit(X_train_norm[perm], y_train_arr[perm], classes=clases)

    # Evaluación en Train y Test al terminar la época
    dec_train = svm_epocas.decision_function(X_train_norm)
    dec_test = svm_epocas.decision_function(X_test_norm)

    acc_tr = accuracy_score(y_train_arr, svm_epocas.predict(X_train_norm)) * 100.0
    acc_te = accuracy_score(y_test_arr, svm_epocas.predict(X_test_norm)) * 100.0
    loss_tr = hinge_loss(y_train_arr, dec_train, labels=clases)
    loss_te = hinge_loss(y_test_arr, dec_test, labels=clases)

    hist_train_acc.append(acc_tr)
    hist_test_acc.append(acc_te)
    hist_train_loss.append(loss_tr)
    hist_test_loss.append(loss_te)

    if (acc_te / 100.0) > (mejor_test_acc + TOL_MEJORA):
        mejor_test_acc = acc_te / 100.0
        mejor_epoca = epoca
        sin_mejora = 0
    else:
        sin_mejora += 1
        if epoca_tope is None and sin_mejora >= PACIENCIA:
            epoca_tope = epoca

    if epoca == 1 or epoca % 5 == 0 or epoca == MAX_EPOCAS:
        print(f"  Época {epoca:2d}/{MAX_EPOCAS} | Train Acc: {acc_tr:.2f}% | Test Acc: {acc_te:.2f}% | Train Loss: {loss_tr:.4f} | Test Loss: {loss_te:.4f}")

print("\n" + "-" * 60)
print(f"[MEJOR RESULTADO] Exactitud máxima en Test: {mejor_test_acc * 100:.2f}% (Época #{mejor_epoca})")
if epoca_tope:
    print(f"[TOPE DETECTADO] Dejó de mejorar alrededor de la época #{epoca_tope} ({PACIENCIA} épocas sin avance >= {TOL_MEJORA*100:.2f}%).")
else:
    print("[INFO] El modelo siguió mejorando hasta el final; aumenta MAX_EPOCAS para explorar más.")
print("-" * 60)

# Gráfico dual: Accuracy y Loss por Época (Train vs. Test)
epocas_x = np.arange(1, MAX_EPOCAS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Exactitud (Accuracy)
ax1.plot(epocas_x, hist_train_acc, color='#1a73e8', linewidth=2, label='Train Accuracy')
ax1.plot(epocas_x, hist_test_acc, color='#1e8e3e', linewidth=2, label='Test Accuracy')
ax1.axvline(mejor_epoca, color='#e37400', linestyle='--', linewidth=1.5, label=f'Mejor Test (Época {mejor_epoca})')
ax1.set_title('Exactitud por Época (Train vs. Test)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Época')
ax1.set_ylabel('Exactitud (%)')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower right')

# Panel 2: Pérdida (Hinge Loss)
ax2.plot(epocas_x, hist_train_loss, color='#1a73e8', linewidth=2, label='Train Loss')
ax2.plot(epocas_x, hist_test_loss, color='#d93025', linewidth=2, label='Test Loss')
ax2.axvline(mejor_epoca, color='#e37400', linestyle='--', linewidth=1.5, label=f'Mejor Test (Época {mejor_epoca})')
ax2.set_title('Hinge Loss por Época (Train vs. Test)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida (menor = mejor)')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='upper right')

plt.suptitle('Curva de Aprendizaje por Épocas — SVM Lineal (SGD Hinge)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('img/curva_epocas_svm.png', dpi=150)
plt.show()
print("[OK] Guardada en 'img/curva_epocas_svm.png'.")

### Celda 9: Prueba Unitaria (Muestra #1100)

In [ ]:
idx_demo = 1100
muestra_norm = X_test_norm[idx_demo]
muestra_raw = X_test.iloc[idx_demo].values
real = int(y_test_arr[idx_demo])
pred = int(modelo.predict([muestra_norm])[0])

plt.figure(figsize=(3.5, 3.5))
plt.imshow(muestra_raw.reshape(28, 28), cmap='gray')
estado = 'ACIERTO' if pred == real else 'ERROR'
plt.title(f"Índice #{idx_demo}\nReal: {real} | Predicción: {pred} ({estado})", fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('img/muestra_individual_1100.png', dpi=150)
plt.show()

print(f"Muestra #{idx_demo} -> Real: {real} | Predicho por SVM: {pred} -> {estado}")

### Celda 10: Empaquetar y Descargar Artefactos (Modelo, Métricas y Gráficos)

In [ ]:
os.makedirs('modelos/svm', exist_ok=True)

ruta_modelo = 'modelos/svm/modelo_svm.joblib'
ruta_test = 'modelos/datos_test.npz'
ruta_metricas = 'modelos/svm/metricas_svm.json'
ruta_info = 'modelos/svm/info.json'

# 1. Guardar modelo SVC entrenado
joblib.dump(modelo, ruta_modelo)

# 2. Guardar conjunto de prueba (píxeles uint8 para ahorrar espacio)
np.savez_compressed(
    ruta_test,
    X_test=X_test.values.astype(np.uint8),
    y_test=y_test_arr.astype(np.int64),
    y_pred=y_pred.astype(np.int64)
)

# 3. Guardar métricas completas (incluyendo Accuracy Train vs. Test)
indices_error = np.where(y_test_arr != y_pred)[0].tolist()
metricas = {
    'modelo': "SVM (kernel='linear') [Normalizado]",
    'semilla': SEMILLA,
    'normalizado': True,
    'total_muestras': int(len(df)),
    'muestras_entrenamiento': int(len(X_train)),
    'muestras_prueba': int(len(X_test)),
    'accuracy_train': float(precision_train),
    'accuracy': float(precision),
    'tiempo_entrenamiento_seg': round(tiempo_train, 2),
    'tiempo_prediccion_seg': round(tiempo_pred, 2),
    'aciertos': int(len(y_test_arr) - len(indices_error)),
    'errores': int(len(indices_error)),
    'indices_error': indices_error,
    'epocas_sgd_mejor_epoca': int(mejor_epoca),
    'epocas_sgd_mejor_test_acc': round(mejor_test_acc * 100, 2)
}
with open(ruta_metricas, 'w', encoding='utf-8') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

# 4. Guardar metadatos info.json para el servidor web
info_modelo = {
    'id': 'svm',
    'nombre': 'SVM Lineal',
    'familia': 'Máquina de Soporte Vectorial (SVC)',
    'accuracy_pct': round(precision * 100, 2),
    'archivo_modelo': 'modelo_svm.joblib',
    'archivo_metricas': 'metricas_svm.json',
    'normalizado': True
}
with open(ruta_info, 'w', encoding='utf-8') as f:
    json.dump(info_modelo, f, indent=2, ensure_ascii=False)

# 5. Empaquetar modelo, métricas, datos de test y todos los gráficos de img/
zip_salida = 'artefactos_detectia.zip'
with zipfile.ZipFile(zip_salida, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(ruta_modelo, arcname=ruta_modelo)
    z.write(ruta_test, arcname=ruta_test)
    z.write(ruta_metricas, arcname=ruta_metricas)
    z.write(ruta_info, arcname=ruta_info)
    for img_name in sorted(os.listdir('img')):
        ruta_img = os.path.join('img', img_name)
        if os.path.isfile(ruta_img):
            z.write(ruta_img, arcname=os.path.join('img', img_name))

print(f"\n[OK] Paquete creado: {zip_salida} ({os.path.getsize(zip_salida) / (1024*1024):.2f} MB)")
print("Imágenes incluidas en el ZIP:", sorted(os.listdir('img')))
print("Iniciando descarga automática...")
files.download(zip_salida)

print("\n" + "="*60)
print("Proceso finalizado. Pasos en tu entorno local:")
print("1. Descomprime 'artefactos_detectia.zip' en la raíz de 'DetectIA'.")
print("2. (Opcional) Para actualizar la app Android, ejecuta:")
print("   python modelos/svm/exportar_modelo_android.py")
print("3. Para abrir el laboratorio web, ejecuta:")
print("   python web/app.py")
print("="*60)